# MATH 550 Final Project — Phase 1 (Track A vLLM: Step 1B Pilot v2 & Step 1C Full Cache)

**Research Question:** *When should an LLM self-refine, sample more, think longer, or escalate to a larger model?*

### How to Run (Colab GPU Only: L4 22.5GB / A100 / G4)
1. Connect this notebook to a **Google Colab GPU runtime** (`L4`, `A100`, or `G4`; `T4` cannot hold unquantized `Qwen3-8B`).
2. **Step 1B (Pilot 50 v2, ~3–5 min):** Keep `RUN_MODE = "pilot"` in Cell 1 and click **Run All**. Verify low Non-Thinking truncation (`max_new_tokens=2048`), clean `vLLM` log-probs, and download `math550_pilot_v2_bundle.zip`.
3. **Step 1C (Lean Full Cache, 5,600 gens):** Change `RUN_MODE = "full"` in Cell 1 and click **Run All**. Prioritizes `math500_test_500 (Atom 1-3)` -> `router_val_200` & stratified `router_train_600` (`core 300` then `extra 300`) -> `Atom 4 (8B)` -> `Atom 5 (1.7B Thinking)` last, with continuous-batch chunks saved to Google Drive.

> **Note:** Cell 1 & Cell 2 intentionally run `vLLM` setup *before* importing `torch`.

In [ ]:
# === Cell 1: Pre-Import Colab Guard & Experiment Config (Do NOT import torch yet!) ===
import importlib.util
import os
import subprocess
from pathlib import Path

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["VLLM_WORKER_MULTIPROCESS_METHOD"] = "spawn"

ALLOW_NON_COLAB_OVERRIDE = False
HAS_COLAB_PKG = importlib.util.find_spec("google.colab") is not None
IN_COLAB = HAS_COLAB_PKG or Path("/content").exists()
if not IN_COLAB and not ALLOW_NON_COLAB_OVERRIDE:
    raise RuntimeError("Non-Colab runtime detected. Per project rules, run this notebook on Google Colab.")

gpu_query = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
if gpu_query.returncode != 0:
    raise RuntimeError(f"CUDA GPU unavailable via nvidia-smi: {gpu_query.stderr}")
print("Detected GPU(s):", gpu_query.stdout.strip())

# --- Experiment Mode & Version ---
# Keep "pilot" for Step 1B (pilot_50_v2), then switch to "full" for Step 1C.
RUN_MODE = "pilot"  # choices: "pilot", "full"
RUN_TAG = "v2"
ENGINE_NAME = "vllm"
TRY_MOUNT_GOOGLE_DRIVE = True
RUN_BS1_LATENCY_CALIBRATION = True
CALIBRATION_SAMPLE_COUNT = 10
CALIBRATION_THINK_COUNT = 5

# --- Models, Limits & Decoding Contract ---
MODEL_SMALL = "Qwen/Qwen3-1.7B"
MODEL_LARGE = "Qwen/Qwen3-8B"
MAX_NEW_TOKENS_NON_THINK = 2048
MAX_NEW_TOKENS_CRITIQUE = 512
MAX_NEW_TOKENS_THINK = 4096
N_SAMPLING_PATHS = 3
SAMPLING_PARAMS_NON_THINK = {"temperature": 0.7, "top_p": 0.8, "top_k": 20}
SAMPLING_PARAMS_THINK = {"temperature": 0.6, "top_p": 0.95, "top_k": 20}
CHUNK_SIZE_SMALL = 250
CHUNK_SIZE_LARGE = 200
SEED = 550


In [ ]:
# === Cell 2: Verify/Install vLLM & Dependencies (Pre-Torch) + Mount Google Drive ===
import importlib.metadata
import subprocess
import sys

def _ver_tuple(pkg: str) -> tuple[int, ...]:
    try:
        raw = importlib.metadata.version(pkg).split("+")[0]
        parts = []
        for seg in raw.split(".")[:3]:
            digits = "".join(ch for ch in seg if ch.isdigit())
            parts.append(int(digits) if digits else 0)
        return tuple(parts)
    except importlib.metadata.PackageNotFoundError:
        return (0, 0, 0)

def _check_vllm_subprocess() -> tuple[str, str, str]:
    code = "\n".join([
        "import importlib.metadata as md, importlib.util, torch",
        "vllm_ver = 'MISSING'",
        "if importlib.util.find_spec('vllm') is not None:",
        "    import vllm",
        "    vllm_ver = getattr(vllm, '__version__', md.version('vllm'))",
        "print(f'{torch.__version__}|{torch.version.cuda}|{vllm_ver}')",
    ])
    res = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
    if res.returncode != 0:
        return ("ERROR", res.stderr.strip(), res.stdout.strip())
    parts = res.stdout.strip().splitlines()[-1].split("|")
    return (parts[0], parts[1], parts[2])

torch_before, cuda_before, vllm_before = _check_vllm_subprocess()
print(f"Environment check -> PyTorch={torch_before} | CUDA={cuda_before} | vLLM={vllm_before}")

need_install = []
if vllm_before in ("MISSING", "ERROR"):
    need_install.append("vllm")
if _ver_tuple("transformers") < (4, 51, 0):
    need_install.append("transformers>=4.51.0")
if _ver_tuple("sympy") < (1, 12, 0):
    need_install.append("sympy>=1.12")

if need_install:
    print("Installing required packages:", need_install)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *need_install])
    torch_after, cuda_after, vllm_after = _check_vllm_subprocess()
    print(f"After install     -> PyTorch={torch_after} | CUDA={cuda_after} | vLLM={vllm_after}")
    if torch_after == "ERROR":
        raise RuntimeError(f"vLLM import check failed after install: {cuda_after}")
    if torch_after != torch_before:
        print(
            "NOTE: PyTorch changed during install. If Cell 5 hits any shared-library error, "
            "click Runtime -> Restart session once and re-run from Cell 1."
        )
else:
    print("All required packages already satisfy minimum versions.")

import torch
import transformers
import vllm

if not torch.cuda.is_available() and not ALLOW_NON_COLAB_OVERRIDE:
    raise RuntimeError("torch.cuda.is_available() is False.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
major_cc = torch.cuda.get_device_capability(0)[0]
DTYPE_STR = "bfloat16" if (major_cc >= 8 and torch.cuda.is_bf16_supported()) else "float16"

print(
    f"Active GPU={gpu_name} | VRAM={vram_gb:.1f} GB | dtype={DTYPE_STR} | "
    f"transformers={transformers.__version__} | vllm={getattr(vllm, '__version__', 'ok')}"
)
if vram_gb < 18.5:
    raise RuntimeError(
        f"Active GPU only has {vram_gb:.1f} GB VRAM (likely T4). "
        "Unquantized Qwen3-8B bfloat16 requires >=22 GB (switch Colab GPU to L4, A100, or G4)."
    )

OUTPUT_ROOT = Path("/content/math550_outputs") if Path("/content").exists() else Path("./outputs")
if TRY_MOUNT_GOOGLE_DRIVE and HAS_COLAB_PKG:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        OUTPUT_ROOT = Path("/content/drive/MyDrive/math550_project_outputs")
        print(f"Google Drive mounted. Persistent OUTPUT_ROOT = {OUTPUT_ROOT}")
    except Exception as exc:
        print(f"Drive mount skipped/unavailable ({exc}); using local VM path {OUTPUT_ROOT}")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
# === Cell 3: Verified Project Utilities (Grading, Splits, Feature Math, Resume I/O) ===
from __future__ import annotations

"""Math answer extraction, SymPy equivalence grading, and 4-state reliability auditing.

Adapted cleanly from the course evaluation utilities (`reasoning_from_scratch.ch03` /
`demo_base_model_evaluation.ipynb`) with explicit 4-state classification
(`correct`, `wrong`, `invalid`, `truncated`) and truncation awareness.
"""


from dataclasses import asdict, dataclass
import re
from typing import Literal

from sympy import simplify
from sympy.core.sympify import SympifyError
from sympy.parsing import sympy_parser as spp
from sympy.polys.polyerrors import PolynomialError
from tokenize import TokenError


OutcomeStatus = Literal["correct", "wrong", "invalid", "truncated"]

RE_NUMBER = re.compile(r"-?(?:\d+/\d+|\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)")
RE_SPECIAL = re.compile(r"<\|[^>]+?\|>")

LATEX_FIXES = [
    (r"\\left\s*", ""),
    (r"\\right\s*", ""),
    (r"\\,|\\!|\\;|\\:", ""),
    (r"~|\\\s", " "),
    (r"\\cdot", "*"),
    ("\u00b7|\u00d7", "*"),
    (r"\\\^\\circ", ""),
    (r"\\dfrac", r"\\frac"),
    (r"\\tfrac", r"\\frac"),
    ("°", ""),
]

UNIT_TEXT_RE = re.compile(
    r"\\(?:text|mbox)\s*\{([\sA-Za-z.,/\-]*[A-Za-z][\sA-Za-z.,/\-]*)\}(?:\s*\^\s*\{?\d+\}?)?"
)

STYLE_WRAPPER_RE = re.compile(r"\\(?:mathbf|mathrm|textbf|boldsymbol|mathit)\s*\{([^{}]*)\}")

MATRIX_RE = re.compile(
    r"\\begin\{(?:p|b|v|V)?matrix\}(.*?)\\end\{(?:p|b|v|V)?matrix\}",
    re.DOTALL,
)

TRIG_LOG_RE = re.compile(
    r"\\(arcsin|arccos|arctan|sin|cos|tan|cot|sec|csc|log|ln|exp)(?![A-Za-z])"
)

SINGLE_VAR_ASSIGN_RE = re.compile(
    r"^(?:\\mathbf\s*\{)?([A-Za-z])(?:\})?\s*=\s*(.+)$"
)

SUPERSCRIPT_MAP = {
    "⁰": "0",
    "¹": "1",
    "²": "2",
    "³": "3",
    "⁴": "4",
    "⁵": "5",
    "⁶": "6",
    "⁷": "7",
    "⁸": "8",
    "⁹": "9",
    "⁺": "+",
    "⁻": "-",
    "⁽": "(",
    "⁾": ")",
}


@dataclass(frozen=True)
class GradeResult:
    """Structured outcome for a single model completion."""

    status: OutcomeStatus
    correct: bool
    extracted_correct_ignoring_truncation: bool
    has_boxed: bool
    extracted_answer: str | None
    truncated: bool
    think_closed: bool | None = None

    def to_dict(self) -> dict[str, object]:
        return asdict(self)


def split_thinking_response(text: str, *, think_in_prompt: bool = False) -> tuple[str, bool]:
    """Split Qwen3 `<think>...</think>` reasoning from the final response body.

    Returns `(final_body, think_closed)`. When `think_in_prompt=True` (the Qwen3 chat
    template appends `<think>` to the prompt), a missing `</think>` means the thinking
    block was never closed (`think_closed=False`).
    """

    if not text:
        return "", not think_in_prompt
    close_tag = "</think>"
    if close_tag in text:
        idx = text.rfind(close_tag)
        return text[idx + len(close_tag) :].strip(), True
    if think_in_prompt or "<think>" in text:
        return "", False
    return text, True


def get_last_boxed_with_span(text: str) -> tuple[str, int, int] | None:
    """Return `(content, content_start, content_end)` for the last `\\boxed{...}`."""

    if not text:
        return None
    boxed_start_idx = text.rfind(r"\boxed")
    if boxed_start_idx == -1:
        return None

    current_idx = boxed_start_idx + len(r"\boxed")
    while current_idx < len(text) and text[current_idx].isspace():
        current_idx += 1

    if current_idx >= len(text) or text[current_idx] != "{":
        return None

    current_idx += 1
    brace_depth = 1
    content_start_idx = current_idx

    while current_idx < len(text) and brace_depth > 0:
        char = text[current_idx]
        if char == "{":
            brace_depth += 1
        elif char == "}":
            brace_depth -= 1
        current_idx += 1

    if brace_depth != 0:
        return None

    content_end_idx = current_idx - 1
    return text[content_start_idx:content_end_idx], content_start_idx, content_end_idx


def get_last_boxed(text: str) -> str | None:
    """Return the inner content of the last balanced `\\boxed{...}` expression."""

    found = get_last_boxed_with_span(text)
    return found[0] if found is not None else None


def extract_final_candidate(text: str, fallback: str = "number_only") -> str:
    """Extract the final candidate expression from a completion.

    Prefers the last `\\boxed{...}` expression. When `fallback` is `'number_only'`,
    falls back to the last numeric token and returns `""` if none exists so unparseable
    prose can be classified as `invalid`.
    """

    if not text:
        return ""
    boxed = get_last_boxed(text.strip())
    if boxed is not None and boxed.strip():
        return boxed.strip().strip("$ ")

    if fallback in ("number_then_full", "number_only"):
        matches = RE_NUMBER.findall(text)
        if matches:
            return matches[-1]
        if fallback == "number_then_full":
            return text.strip()
    return ""


def _normalize_latex_fractions_and_roots(text: str) -> str:
    """Canonicalize LaTeX fraction/root shorthands and mixed numbers before SymPy parsing."""

    # Separate `\sqrt` from any immediately preceding alphanumeric/bracket token (e.g. `30p\sqrt{p}`).
    text = re.sub(r"(?<=[0-9A-Za-z\)\]\}])(?=\\sqrt(?![A-Za-z]))", " ", text)

    # Normalize `\sqrt` variants (`\sqrt{x}`, `\sqrt 2`, `\sqrt2`).
    for _ in range(4):
        text = re.sub(r"\\sqrt\s*\{([^{}]*)\}", lambda m: f"sqrt({m.group(1)})", text)
    text = re.sub(r"\\sqrt\s+([0-9A-Za-z]+)", lambda m: f"sqrt({m.group(1)})", text)
    text = re.sub(r"\\sqrt([0-9A-Za-z])", lambda m: f"sqrt({m.group(1)})", text)

    # Canonicalize shorthand `\frac` variants (`\frac12`, `\frac 32`, `\frac9{19}`, `\frac{50}3`).
    for _ in range(4):
        text = re.sub(
            r"\\frac\s*\{([^{}]+)\}\s*([0-9A-Za-z])(?![0-9A-Za-z{])",
            lambda m: f"\\frac{{{m.group(1)}}}{{{m.group(2)}}}",
            text,
        )
        text = re.sub(
            r"\\frac\s*([0-9A-Za-z])\s*\{([^{}]+)\}",
            lambda m: f"\\frac{{{m.group(1)}}}{{{m.group(2)}}}",
            text,
        )
        text = re.sub(
            r"\\frac\s*([0-9A-Za-z])\s*([0-9A-Za-z])",
            lambda m: f"\\frac{{{m.group(1)}}}{{{m.group(2)}}}",
            text,
        )

    # Convert mixed numbers like `6\frac{1}{5}` or `-3\frac{1}{2}` to `(6+(1)/(5))` / `(-(3+(1)/(2)))`.
    text = re.sub(
        r"(?<![A-Za-z0-9._^])(-?)(\d+)\s*\\frac\s*\{(\d+)\}\s*\{(\d+)\}",
        lambda m: (
            f"(-({m.group(2)}+({m.group(3)})/({m.group(4)})))"
            if m.group(1) == "-"
            else f"({m.group(2)}+({m.group(3)})/({m.group(4)}))"
        ),
        text,
    )

    # Expand nested `\frac{num}{den}` inner-to-outer.
    for _ in range(5):
        updated = re.sub(
            r"\\frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}",
            lambda m: f"({m.group(1)})/({m.group(2)})",
            text,
        )
        if updated == text:
            break
        text = updated

    text = re.sub(
        r"\\frac\s+([0-9A-Za-z]+)\s+([0-9A-Za-z]+)",
        lambda m: f"({m.group(1)})/({m.group(2)})",
        text,
    )
    return text


def _strip_unit_and_style_wrappers(text: str) -> str:
    """Remove styling macros and trailing alphabetic units without harming standalone words."""

    for _ in range(3):
        updated = STYLE_WRAPPER_RE.sub(lambda m: m.group(1), text)
        if updated == text:
            break
        text = updated

    match = re.match(r"^\\(?:text|mbox)\s*\{(?P<x>.+?)\}$", text)
    if match:
        text = match.group("x").strip()

    # Strip unit annotations like `\text{ days}` or `\mbox{ inches}^2` when math remains.
    stripped_units = UNIT_TEXT_RE.sub("", text).strip()
    if stripped_units and re.search(r"[0-9]", stripped_units):
        text = stripped_units

    # Unwrap any remaining `\text{...}` / `\mbox{...}` (e.g., `(\text{E})` -> `(E)`).
    text = re.sub(r"\\(?:text|mbox)\s*\{([^{}]*)\}", lambda m: m.group(1), text)
    return text


def _normalize_matrices(text: str) -> str:
    """Canonicalize LaTeX matrix environments to `[[row1];[row2]]` representation."""

    def _repl(match: re.Match[str]) -> str:
        body = match.group(1).strip()
        rows = [r.strip() for r in re.split(r"\\\\", body) if r.strip()]
        norm_rows = []
        for row in rows:
            cols = [normalize_text(c.strip()) for c in row.split("&")]
            norm_rows.append("[" + ",".join(cols) + "]")
        return "matrix(" + ";".join(norm_rows) + ")"

    return MATRIX_RE.sub(_repl, text)


def _strip_plain_thousand_separators(text: str) -> str:
    """Remove ASCII thousand-separator commas (e.g. `1,234/2`, `1,234.56`) at bracket depth 0.

    Leaves commas inside parentheses, brackets, or braces (`(12,102)`, `(1,234)`,
    `[100,200]`, `{1,234}`) untouched so interval, tuple, and set elements are never merged.
    """

    if "," not in text:
        return text

    depths: list[int] = []
    depth = 0
    for ch in text:
        if ch in "([{":
            depth += 1
            depths.append(depth)
        elif ch in ")]}":
            depths.append(depth)
            depth = max(0, depth - 1)
        else:
            depths.append(depth)

    remove_indices: set[int] = set()
    for match in re.finditer(r"(?<![0-9.,])\d{1,3}(?:,\d{3})+(?=(?:\.\d+)?(?![0-9.,]))", text):
        start, end = match.span()
        if all(depths[idx] == 0 for idx in range(start, end)):
            for idx in range(start, end):
                if text[idx] == ",":
                    remove_indices.add(idx)

    if not remove_indices:
        return text
    return "".join(ch for idx, ch in enumerate(text) if idx not in remove_indices)


def normalize_text(text: str | None) -> str:
    """Normalize LaTeX/text answer representations before symbolic comparison."""

    if not text:
        return ""
    text = RE_SPECIAL.sub("", str(text)).strip()

    match = re.match(r"^[A-Za-z]\s*[.:]\s*(.+)$", text)
    if match:
        text = match.group(1)

    text = re.sub(r"\^\s*\{\s*\\circ\s*\}", "", text)
    text = re.sub(r"\^\s*\\circ", "", text)
    text = text.replace("°", "")

    text = re.sub(r"\\\(|\\\)|\\\[|\\\]", "", text)

    # Parse matrix environments BEFORE LATEX_FIXES so `\\ ` row separators stay intact.
    if MATRIX_RE.search(text):
        text = _normalize_matrices(text)

    # Strip explicit LaTeX digit-grouping commas (`11,\! 111,\! 111,\! 100`, `14{,}916`) before `\,`/`\!` removal.
    text = re.sub(r"(?<=\d)(?:,(?:\\[,!;:]\s*)+|\{,\})(?=\d{3}(?!\d))", "", text)

    for pat, rep in LATEX_FIXES:
        text = re.sub(pat, rep, text)

    text = _strip_unit_and_style_wrappers(text)

    def convert_superscripts(s: str, base: str | None = None) -> str:
        converted = "".join(SUPERSCRIPT_MAP.get(ch, ch) for ch in s)
        if base is None:
            return converted
        return f"{base}**{converted}"

    text = re.sub(
        r"([0-9A-Za-z\)\]\}])([⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻]+)",
        lambda m: convert_superscripts(m.group(2), base=m.group(1)),
        text,
    )
    text = convert_superscripts(text)

    text = text.replace("\\%", "%").replace("\\$", "").replace("$", "").replace("%", "")
    # Preserve escaped set braces as `{` / `}` until after `_strip_plain_thousand_separators`.
    text = text.replace(r"\{", "{").replace(r"\}", "}")
    text = _normalize_latex_fractions_and_roots(text)

    # Canonicalize common symbolic constants and functions for SymPy parsing.
    text = re.sub(r"\\infty\b", " oo ", text)
    text = re.sub(r"\\pi(?![A-Za-z])", " pi ", text)
    text = TRIG_LOG_RE.sub(lambda m: f" {m.group(1)} ", text)

    text = text.replace("^", "**")
    text = re.sub(r"(?<=\d)\s+(\d+/\d+)", lambda m: "+" + m.group(1), text)
    text = _strip_plain_thousand_separators(text)
    # Remove leading zeros on multi-digit integers (`049` -> `49`) while preserving `0.5`.
    text = re.sub(r"(?<![0-9.])0+(?=\d+(?!\d*\.))", "", text)

    # Strip spaces around operators slightly while keeping multiplication tokens clear.
    text = re.sub(r"\s+", " ", text)
    return text.replace("{", "").replace("}", "").strip().lower()


def sympy_parser(expr: str | None):
    """Parse normalized math text into a SymPy expression, returning None on failure."""

    if expr is None or not expr or len(expr) > 2000:
        return None
    try:
        return spp.parse_expr(
            expr,
            transformations=(
                *spp.standard_transformations,
                spp.implicit_multiplication_application,
            ),
            evaluate=True,
        )
    except (
        SympifyError,
        SyntaxError,
        TypeError,
        AttributeError,
        IndexError,
        TokenError,
        ValueError,
        PolynomialError,
        RecursionError,
    ):
        return None


def _parse_matrix_rows(norm_text: str) -> list[list[str]] | None:
    match = re.fullmatch(r"matrix\((.+)\)", norm_text.strip())
    if not match:
        return None
    rows = []
    for row_str in match.group(1).split(";"):
        row_str = row_str.strip()
        if not (row_str.startswith("[") and row_str.endswith("]")):
            return None
        rows.append([cell.strip() for cell in row_str[1:-1].split(",")])
    return rows


def _equations_equivalent(expr_gtruth: str, expr_pred: str) -> bool:
    if "=" not in expr_gtruth or "=" not in expr_pred:
        return False
    gt_sides = [s.strip() for s in expr_gtruth.split("=")]
    pr_sides = [s.strip() for s in expr_pred.split("=")]
    if len(gt_sides) != 2 or len(pr_sides) != 2:
        return False
    gt_l, gt_r = sympy_parser(gt_sides[0]), sympy_parser(gt_sides[1])
    pr_l, pr_r = sympy_parser(pr_sides[0]), sympy_parser(pr_sides[1])
    if any(x is None for x in (gt_l, gt_r, pr_l, pr_r)):
        return False
    try:
        diff_g = simplify(gt_l - gt_r)
        diff_p = simplify(pr_l - pr_r)
        if diff_g == 0 or diff_p == 0:
            return diff_g == diff_p
        ratio = simplify(diff_g / diff_p)
        return ratio.is_Number and ratio != 0
    except (
        SympifyError,
        TypeError,
        ValueError,
        ZeroDivisionError,
        RecursionError,
        NotImplementedError,
    ):
        return False


def equality_check(expr_gtruth: str, expr_pred: str) -> bool:
    """Return True if two normalized answer expressions are identical or symbolically equal."""

    if expr_gtruth == expr_pred:
        return True
    gt_mat = _parse_matrix_rows(expr_gtruth)
    pr_mat = _parse_matrix_rows(expr_pred)
    if gt_mat is not None and pr_mat is not None:
        if len(gt_mat) != len(pr_mat):
            return False
        return all(
            len(rg) == len(rp) and all(equality_check(cg, cp) for cg, cp in zip(rg, rp))
            for rg, rp in zip(gt_mat, pr_mat)
        )
    if _equations_equivalent(expr_gtruth, expr_pred):
        return True
    gtruth = sympy_parser(expr_gtruth)
    pred = sympy_parser(expr_pred)
    if gtruth is not None and pred is not None:
        if gtruth == pred:
            return True
        try:
            return bool(simplify(gtruth - pred) == 0)
        except (
            SympifyError,
            TypeError,
            ValueError,
            ZeroDivisionError,
            RecursionError,
            NotImplementedError,
        ):
            pass
    return False


def _outer_brackets_wrap_entire(text: str) -> bool:
    if len(text) < 2 or text[0] not in "([" or text[-1] not in ")]":
        return False
    depth = 0
    for idx, ch in enumerate(text):
        if ch in "([":
            depth += 1
        elif ch in ")]":
            depth -= 1
            if depth == 0 and idx < len(text) - 1:
                return False
            if depth < 0:
                return False
    return depth == 0


def _split_top_level_commas(text: str) -> list[str]:
    parts: list[str] = []
    depth = 0
    start = 0
    for idx, ch in enumerate(text):
        if ch in "([":
            depth += 1
        elif ch in ")]":
            depth = max(0, depth - 1)
        elif ch == "," and depth == 0:
            parts.append(text[start:idx].strip())
            start = idx + 1
    parts.append(text[start:].strip())
    return parts


def _expand_pm(parts: list[str]) -> list[str]:
    expanded: list[str] = []
    for part in parts:
        if r"\pm" in part and part.count(r"\pm") == 1:
            lhs, rhs = part.split(r"\pm", 1)
            expanded.append(f"{lhs.strip()} + {rhs.strip()}".strip())
            expanded.append(f"{lhs.strip()} - {rhs.strip()}".strip())
        else:
            expanded.append(part)
    return expanded


def split_into_parts(text: str) -> list[str]:
    """Split union/tuple/list-like answers into ordered components."""

    if not text:
        return []
    if r"\cup" in text:
        union_parts = [p.strip() for p in text.split(r"\cup") if p.strip()]
        if len(union_parts) > 1:
            expanded: list[str] = []
            for part in union_parts:
                expanded.extend(split_into_parts(part))
            return _expand_pm(expanded)
    if _outer_brackets_wrap_entire(text) and "," in text[1:-1]:
        items = _split_top_level_commas(text[1:-1])
        if all(items):
            return _expand_pm(items)
    if "," in text:
        top_items = _split_top_level_commas(text)
        if len(top_items) > 1 and all(top_items):
            return _expand_pm(top_items)
    return _expand_pm([text])


def _strip_single_var_lhs(raw_text: str) -> tuple[str | None, str]:
    stripped = str(raw_text).strip()
    match = SINGLE_VAR_ASSIGN_RE.match(stripped)
    if match:
        return match.group(1).lower(), match.group(2).strip()
    return None, stripped


def _strip_var_membership(raw_text: str) -> tuple[str | None, str]:
    stripped = str(raw_text).strip()
    match = re.match(r"^(?:\\mathbf\s*\{)?([A-Za-z])(?:\})?\s*\\in\s*(.+)$", stripped)
    if match:
        return match.group(1).lower(), match.group(2).strip()
    return None, stripped


def _interval_bracket_signature(text: str) -> tuple[str, ...] | None:
    if r"\cup" in text:
        segs = [s.strip() for s in text.split(r"\cup") if s.strip()]
        sigs = [_interval_bracket_signature(s) for s in segs]
        return tuple(sig[0] for sig in sigs if sig) if all(sigs) else None
    if _outer_brackets_wrap_entire(text) and "," in text[1:-1]:
        return (text[0] + text[-1],)
    return None


def _grade_single(pred_text: str, gt_text: str) -> bool:
    gt_norm = normalize_text(gt_text)
    pr_norm = normalize_text(pred_text)
    gt_sig = _interval_bracket_signature(gt_norm)
    pr_sig = _interval_bracket_signature(pr_norm)
    if gt_sig is not None and pr_sig is not None and gt_sig != pr_sig:
        return False
    gt_parts = split_into_parts(gt_norm)
    pred_parts = split_into_parts(pr_norm)
    if not gt_parts or not pred_parts or len(gt_parts) != len(pred_parts):
        return False
    return all(equality_check(gt, pred) for gt, pred in zip(gt_parts, pred_parts))


def grade_answer(pred_text: str | None, gt_text: str | None) -> bool:
    """Grade predicted candidate text against reference answer text."""

    if pred_text is None or gt_text is None:
        return False
    if _grade_single(pred_text, gt_text):
        return True

    # Handle reference answers formatted with `\approx`, e.g. `\$\dfrac23 \approx \$0.67`.
    if r"\approx" in gt_text:
        for alt in gt_text.split(r"\approx"):
            if alt.strip() and grade_answer(pred_text, alt.strip()):
                return True

    # Handle set membership forms like `x \in [-2, 7]` vs `[-2, 7]`.
    gt_mem_var, gt_mem_rhs = _strip_var_membership(gt_text)
    pr_mem_var, pr_mem_rhs = _strip_var_membership(pred_text)
    if gt_mem_var is not None and pr_mem_var is None:
        if _grade_single(pred_text, gt_mem_rhs):
            return True
    elif pr_mem_var is not None and gt_mem_var is None:
        if _grade_single(pr_mem_rhs, gt_text):
            return True
    elif gt_mem_var is not None and pr_mem_var == gt_mem_var:
        if _grade_single(pr_mem_rhs, gt_mem_rhs):
            return True

    # Handle single-variable assignment forms like `k = \frac{12}{7}` vs `\frac{12}{7}`.
    gt_var, gt_rhs = _strip_single_var_lhs(gt_text)
    pr_var, pr_rhs = _strip_single_var_lhs(pred_text)
    if gt_var is not None and pr_var is None:
        return _grade_single(pred_text, gt_rhs)
    if pr_var is not None and gt_var is None:
        return _grade_single(pr_rhs, gt_text)
    if gt_var is not None and pr_var == gt_var:
        return _grade_single(pr_rhs, gt_rhs)

    return False


def classify_response(
    text: str,
    ground_truth: str,
    *,
    truncated: bool = False,
    strip_thinking: bool = False,
    think_closed: bool | None = None,
) -> GradeResult:
    """Classify a completion into `{correct, wrong, invalid, truncated}`.

    Truncated responses always receive `status="truncated"` and `correct=False` under
    the primary evaluation contract, while preserving `extracted_correct_ignoring_truncation`
    for diagnostic auditing.
    """

    eval_text = text or ""
    if strip_thinking and think_closed is None:
        eval_text, think_closed = split_thinking_response(eval_text, think_in_prompt=True)
    elif strip_thinking and think_closed is False:
        eval_text = ""

    has_boxed = get_last_boxed(eval_text) is not None
    candidate = extract_final_candidate(eval_text, fallback="number_only")
    extracted = candidate if candidate else None
    extracted_correct = bool(extracted is not None and grade_answer(extracted, ground_truth))

    if truncated or (strip_thinking and think_closed is False):
        return GradeResult(
            status="truncated",
            correct=False,
            extracted_correct_ignoring_truncation=extracted_correct,
            has_boxed=has_boxed,
            extracted_answer=extracted,
            truncated=True,
            think_closed=think_closed,
        )
    if extracted is None:
        return GradeResult(
            status="invalid",
            correct=False,
            extracted_correct_ignoring_truncation=False,
            has_boxed=has_boxed,
            extracted_answer=None,
            truncated=False,
            think_closed=think_closed,
        )
    if extracted_correct:
        return GradeResult(
            status="correct",
            correct=True,
            extracted_correct_ignoring_truncation=True,
            has_boxed=has_boxed,
            extracted_answer=extracted,
            truncated=False,
            think_closed=think_closed,
        )
    return GradeResult(
        status="wrong",
        correct=False,
        extracted_correct_ignoring_truncation=False,
        has_boxed=has_boxed,
        extracted_answer=extracted,
        truncated=False,
        think_closed=think_closed,
    )

"""Zero-leakage data download, schema normalization, and stratified split generation (Step 1A)."""


from collections import Counter
import hashlib
import json
from pathlib import Path
import random
import re
import shutil
from typing import Any
from urllib.request import Request, urlopen


MATH_TRAIN_PRIMARY_URL = (
    "https://raw.githubusercontent.com/rasbt/math_full_minus_math500/"
    "refs/heads/main/math_full_minus_math500.json"
)
MATH_TRAIN_BACKUP_URL = (
    "https://f001.backblazeb2.com/file/reasoning-from-scratch/MATH/math_full_minus_math500.json"
)
MATH500_PRIMARY_URL = (
    "https://raw.githubusercontent.com/rasbt/reasoning-from-scratch/"
    "main/ch03/01_main-chapter-code/math500_test.json"
)

DEFAULT_SEED = 550
PILOT_COUNTS = {1: 10, 2: 10, 3: 10, 4: 10, 5: 10}
ROUTER_TRAIN_COUNTS = {1: 52, 2: 108, 3: 126, 4: 154, 5: 160}
ROUTER_VAL_COUNTS = {1: 17, 2: 36, 3: 42, 4: 51, 5: 54}

_USER_AGENT = "MATH550-Final-Project/1.0"


def normalize_problem_text(text: str) -> str:
    """Canonicalize problem whitespace and casing for leakage/duplicate checks."""

    return re.sub(r"\s+", " ", str(text).strip().lower())


def parse_level(raw_level: object) -> int | None:
    """Convert `'Level 3'` or `3` into integer `1..5`, returning None if invalid."""

    if isinstance(raw_level, bool):
        return None
    if isinstance(raw_level, int):
        return raw_level if 1 <= raw_level <= 5 else None
    text = str(raw_level).strip()
    match = re.fullmatch(r"(?:Level\s*)?([1-5])", text, re.IGNORECASE)
    return int(match.group(1)) if match else None


def sha256_bytes(payload: bytes) -> str:
    """Return hexadecimal SHA-256 digest of bytes."""

    return hashlib.sha256(payload).hexdigest()


def sha256_file(path: Path) -> str:
    """Return hexadecimal SHA-256 digest of a file."""

    return sha256_bytes(path.read_bytes())


def _download_with_fallback(urls: list[str], dest: Path) -> str:
    last_error: Exception | None = None
    for url in urls:
        try:
            req = Request(url, headers={"User-Agent": _USER_AGENT})
            with urlopen(req, timeout=60) as resp:
                payload = resp.read()
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(payload)
            return url
        except Exception as exc:  # noqa: BLE001
            last_error = exc
    raise RuntimeError(f"failed to download from {urls}: {last_error}")


def ensure_raw_datasets(
    raw_dir: Path,
    *,
    course_math500_path: Path | None = None,
    force: bool = False,
) -> dict[str, dict[str, object]]:
    """Ensure both `math_full_minus_math500.json` and `math500_test.json` exist locally."""

    raw_dir.mkdir(parents=True, exist_ok=True)
    metadata: dict[str, dict[str, object]] = {}

    train_path = raw_dir / "math_full_minus_math500.json"
    if force or not train_path.is_file():
        src_url = _download_with_fallback([MATH_TRAIN_PRIMARY_URL, MATH_TRAIN_BACKUP_URL], train_path)
        train_status = f"downloaded:{src_url}"
    else:
        train_status = "existing"
    metadata["math_full_minus_math500"] = {
        "path": str(train_path),
        "bytes": train_path.stat().st_size,
        "sha256": sha256_file(train_path),
        "status": train_status,
    }

    test_path = raw_dir / "math500_test.json"
    if force or not test_path.is_file():
        if course_math500_path is not None and course_math500_path.is_file():
            shutil.copy2(course_math500_path, test_path)
            test_status = f"copied:{course_math500_path}"
        else:
            src_url = _download_with_fallback([MATH500_PRIMARY_URL], test_path)
            test_status = f"downloaded:{src_url}"
    else:
        test_status = "existing"
    metadata["math500_test"] = {
        "path": str(test_path),
        "bytes": test_path.stat().st_size,
        "sha256": sha256_file(test_path),
        "status": test_status,
    }
    return metadata


def normalize_records(
    raw_rows: list[dict[str, Any]],
    *,
    source_split: str,
    drop_invalid: bool = True,
) -> tuple[list[dict[str, Any]], dict[str, int]]:
    """Normalize raw MATH / MATH-500 rows to the unified project schema and deduplicate."""

    normalized: list[dict[str, Any]] = []
    seen_problems: set[str] = set()
    stats = {
        "raw_rows": len(raw_rows),
        "dropped_invalid_level": 0,
        "dropped_empty_field": 0,
        "dropped_duplicate_problem": 0,
    }

    for idx, row in enumerate(raw_rows):
        level = parse_level(row.get("level"))
        if level is None:
            if drop_invalid:
                stats["dropped_invalid_level"] += 1
                continue
            raise ValueError(f"invalid level in {source_split} row {idx}: {row.get('level')!r}")

        problem = str(row.get("problem", "")).strip()
        answer = str(row.get("answer", "")).strip()
        if not problem or not answer:
            if drop_invalid:
                stats["dropped_empty_field"] += 1
                continue
            raise ValueError(f"empty problem/answer in {source_split} row {idx}")

        norm_prob = normalize_problem_text(problem)
        if norm_prob in seen_problems:
            if drop_invalid:
                stats["dropped_duplicate_problem"] += 1
                continue
            raise ValueError(f"duplicate problem in {source_split} row {idx}")
        seen_problems.add(norm_prob)

        subject = str(row.get("subject") or row.get("type") or "Unknown").strip()
        raw_uid = row.get("unique_id", idx)
        uid = f"{source_split}:{raw_uid}" if source_split == "math_train" else str(raw_uid)
        normalized.append(
            {
                "unique_id": uid,
                "problem": problem,
                "answer": answer,
                "level": level,
                "subject": subject,
                "source_split": source_split,
            }
        )
    return normalized, stats


def _stratified_take(
    pool_by_level: dict[int, list[dict[str, Any]]],
    counts_by_level: dict[int, int],
    rng: random.Random,
    *,
    split_name: str,
) -> list[dict[str, Any]]:
    chosen: list[dict[str, Any]] = []
    for level in sorted(counts_by_level):
        need = counts_by_level[level]
        available = pool_by_level[level]
        if len(available) < need:
            raise ValueError(
                f"insufficient rows for {split_name} level {level}: need {need}, have {len(available)}"
            )
        selected = available[:need]
        del available[:need]
        for row in selected:
            item = dict(row)
            item["split_name"] = split_name
            chosen.append(item)
    rng.shuffle(chosen)
    return chosen


def build_splits(
    raw_train_rows: list[dict[str, Any]],
    raw_test_rows: list[dict[str, Any]],
    *,
    seed: int = DEFAULT_SEED,
) -> tuple[dict[str, list[dict[str, Any]]], dict[str, Any]]:
    """Build mutually disjoint `pilot_50`, `router_train_600`, `router_val_200`, `math500_test_500`."""

    test_rows, test_stats = normalize_records(raw_test_rows, source_split="math500_test", drop_invalid=False)
    test_norm_problems = {normalize_problem_text(row["problem"]) for row in test_rows}
    test_uids = {row["unique_id"] for row in test_rows}

    train_rows, train_stats = normalize_records(raw_train_rows, source_split="math_train", drop_invalid=True)
    # Enforce hard exclusion of any test-overlapping problem just in case an upstream file changes.
    filtered_train: list[dict[str, Any]] = []
    excluded_overlap = 0
    for row in train_rows:
        if normalize_problem_text(row["problem"]) in test_norm_problems or row["unique_id"] in test_uids:
            excluded_overlap += 1
            continue
        filtered_train.append(row)
    train_stats["dropped_test_overlap"] = excluded_overlap

    rng = random.Random(seed)
    pool_by_level: dict[int, list[dict[str, Any]]] = {lv: [] for lv in range(1, 6)}
    for row in filtered_train:
        pool_by_level[row["level"]].append(row)
    for lv in sorted(pool_by_level):
        pool_by_level[lv].sort(key=lambda r: r["unique_id"])
        rng.shuffle(pool_by_level[lv])

    pilot_50 = _stratified_take(pool_by_level, PILOT_COUNTS, rng, split_name="pilot_50")
    router_train_600 = _stratified_take(
        pool_by_level, ROUTER_TRAIN_COUNTS, rng, split_name="router_train_600"
    )
    router_val_200 = _stratified_take(
        pool_by_level, ROUTER_VAL_COUNTS, rng, split_name="router_val_200"
    )
    math500_test_500 = [dict(row, split_name="math500_test_500") for row in test_rows]

    splits = {
        "pilot_50": pilot_50,
        "router_train_600": router_train_600,
        "router_val_200": router_val_200,
        "math500_test_500": math500_test_500,
    }

    # Explicit cross-split leakage & disjointness audit.
    all_uid_sets = {name: {r["unique_id"] for r in rows} for name, rows in splits.items()}
    all_prob_sets = {
        name: {normalize_problem_text(r["problem"]) for r in rows} for name, rows in splits.items()
    }
    split_names = list(splits)
    pairwise_overlaps: dict[str, dict[str, int]] = {}
    for i, left in enumerate(split_names):
        for right in split_names[i + 1 :]:
            pairwise_overlaps[f"{left}__vs__{right}"] = {
                "unique_id_overlap": len(all_uid_sets[left] & all_uid_sets[right]),
                "normalized_problem_overlap": len(all_prob_sets[left] & all_prob_sets[right]),
            }

    manifest = {
        "seed": seed,
        "cleaning": {"math_train": train_stats, "math500_test": test_stats},
        "pairwise_overlaps": pairwise_overlaps,
        "splits": {
            name: {
                "count": len(rows),
                "levels": dict(sorted(Counter(r["level"] for r in rows).items())),
                "subjects": dict(sorted(Counter(r["subject"] for r in rows).items())),
            }
            for name, rows in splits.items()
        },
    }
    return splits, manifest


def write_splits(
    splits: dict[str, list[dict[str, Any]]],
    manifest: dict[str, Any],
    splits_dir: Path,
    raw_metadata: dict[str, dict[str, object]] | None = None,
) -> Path:
    """Persist split JSONs, combined `splits_bundle.json`, and SHA-256 `splits_manifest.json`."""

    splits_dir.mkdir(parents=True, exist_ok=True)
    file_hashes: dict[str, dict[str, object]] = {}

    for name, rows in splits.items():
        path = splits_dir / f"{name}.json"
        raw_bytes = (json.dumps(rows, ensure_ascii=False, indent=2) + "\n").encode("utf-8")
        path.write_bytes(raw_bytes)
        file_hashes[f"{name}.json"] = {
            "bytes": len(raw_bytes),
            "sha256": sha256_bytes(raw_bytes),
            "count": len(rows),
        }

    bundle_path = splits_dir / "splits_bundle.json"
    bundle_bytes = (json.dumps(splits, ensure_ascii=False, indent=2) + "\n").encode("utf-8")
    bundle_path.write_bytes(bundle_bytes)
    file_hashes["splits_bundle.json"] = {
        "bytes": len(bundle_bytes),
        "sha256": sha256_bytes(bundle_bytes),
    }

    full_manifest = dict(manifest)
    full_manifest["raw_sources"] = raw_metadata or {}
    full_manifest["files"] = file_hashes
    manifest_path = splits_dir / "splits_manifest.json"
    manifest_bytes = (json.dumps(full_manifest, ensure_ascii=False, indent=2) + "\n").encode("utf-8")
    manifest_path.write_bytes(manifest_bytes)
    return manifest_path

"""Pure-Python helpers shared with and tested for the Step 1B Colab collection notebook.

Contains prompt construction, token log-probability feature extraction, boxed-span token
alignment, JSONL atomic checkpoint resume/deduplication, and Pilot diagnostics—without
requiring `torch` or `transformers` to be loaded locally.
"""


import json
import math
from pathlib import Path
from typing import Any, Callable, Sequence



SOLVE_PROMPT_TEMPLATE = (
    "Solve the following math problem step by step. "
    "Put your final answer inside \\boxed{{...}} at the end.\n\n"
    "Problem:\n{problem}"
)

CRITIQUE_PROMPT_TEMPLATE = (
    "You are a careful mathematical reviewer. Check the draft solution step by step "
    "for any logical gaps, algebraic mistakes, or arithmetic errors. "
    "If the draft solution and its final answer are completely correct, explicitly state "
    "'No errors found; the draft is correct.' Do not invent errors and do not use \\boxed{{...}} in your review.\n\n"
    "Problem:\n{problem}\n\n"
    "Draft solution:\n{draft}\n\n"
    "Provide a concise review (under 150 words)."
)

REFINE_PROMPT_TEMPLATE = (
    "Review the problem, the draft solution, and the reviewer's critique below. "
    "If the critique identifies a real error, fix it and provide the corrected solution. "
    "If the draft solution is already correct, confirm and keep the original answer. "
    "Put only the final mathematical answer inside \\boxed{{...}} at the end "
    "(never put words like 'correct' or 'no errors found' inside \\boxed{{...}}).\n\n"
    "Problem:\n{problem}\n\n"
    "Draft solution:\n{draft}\n\n"
    "Critique:\n{critique}"
)


def build_solve_messages(problem: str) -> list[dict[str, str]]:
    """Build chat messages for initial or escalated math solving."""

    return [{"role": "user", "content": SOLVE_PROMPT_TEMPLATE.format(problem=problem)}]


def build_critique_messages(problem: str, draft: str) -> list[dict[str, str]]:
    """Build strictly neutral critique messages that permit confirming correct drafts."""

    return [
        {
            "role": "user",
            "content": CRITIQUE_PROMPT_TEMPLATE.format(problem=problem, draft=draft),
        }
    ]


def build_refine_messages(problem: str, draft: str, critique: str) -> list[dict[str, str]]:
    """Build self-refine messages allowing the model to keep or revise the draft."""

    return [
        {
            "role": "user",
            "content": REFINE_PROMPT_TEMPLATE.format(
                problem=problem, draft=draft, critique=critique
            ),
        }
    ]


def find_boxed_content_token_indices(
    token_ids: Sequence[int],
    decode_tokens: Callable[[Sequence[int]], str],
) -> list[int]:
    """Locate generated token indices overlapping the inner content of the last `\\boxed{...}`.

    Uses cumulative decoding on the tail of `token_ids` so token boundaries match the exact
    generation sequence without re-tokenization drift.
    """

    n = len(token_ids)
    if n == 0:
        return []
    full_text = decode_tokens(token_ids)
    found = get_last_boxed_with_span(full_text)
    if found is None:
        return []
    _, char_start, char_end = found
    if char_end <= char_start:
        return []

    # Find exact character offset intervals for each token via prefix lengths.
    # For efficiency on long outputs, start searching near the end where `\boxed` appears.
    prefix_start_idx = 0
    for probe in range(max(0, n - 128), -1, -1):
        suffix_text = decode_tokens(token_ids[probe:])
        if r"\boxed" in suffix_text:
            prefix_start_idx = probe
            break

    base_prefix_len = len(decode_tokens(token_ids[:prefix_start_idx])) if prefix_start_idx else 0
    prev_len = base_prefix_len
    overlapping: list[int] = []
    for idx in range(prefix_start_idx, n):
        cur_len = len(decode_tokens(token_ids[: idx + 1]))
        tok_start, tok_end = prev_len, cur_len
        if tok_end > char_start and tok_start < char_end:
            overlapping.append(idx)
        prev_len = cur_len
        if tok_start >= char_end:
            break
    return overlapping


def extract_router_features(
    token_logprobs: Sequence[float],
    token_ids: Sequence[int],
    decode_tokens: Callable[[Sequence[int]], str],
    *,
    input_tokens: int,
    truncated: bool,
    response_text: str | None = None,
) -> dict[str, float | int | bool]:
    """Compute the 6 strictly online first-pass features for the logistic router.

    Features:
      1. `mean_token_logprob`
      2. `min_token_logprob`
      3. `answer_logprob`
      4. `output_length`
      5. `format_valid` (`not truncated and has_boxed`)
      6. `question_length` (`input_tokens`)
    """

    if len(token_logprobs) != len(token_ids):
        raise ValueError("token_logprobs and token_ids must have identical length")

    clean_lps = [float(lp) if math.isfinite(float(lp)) else -20.0 for lp in token_logprobs]
    out_len = len(clean_lps)
    if out_len == 0:
        mean_lp = -20.0
        min_lp = -20.0
    else:
        mean_lp = sum(clean_lps) / out_len
        min_lp = min(clean_lps)

    text = response_text if response_text is not None else decode_tokens(token_ids)
    has_boxed = get_last_boxed_with_span(text) is not None
    answer_indices = find_boxed_content_token_indices(token_ids, decode_tokens)
    if answer_indices:
        answer_lp = sum(clean_lps[i] for i in answer_indices) / len(answer_indices)
        answer_span_found = True
    else:
        answer_lp = mean_lp
        answer_span_found = False

    format_valid = float((not truncated) and has_boxed)
    return {
        "mean_token_logprob": float(mean_lp),
        "min_token_logprob": float(min_lp),
        "answer_logprob": float(answer_lp),
        "output_length": int(out_len),
        "format_valid": float(format_valid),
        "question_length": int(input_tokens),
        "has_boxed": bool(has_boxed),
        "answer_span_found": bool(answer_span_found),
    }


def load_completed_ids(jsonl_path: Path) -> set[str]:
    """Read already-checkpointed `unique_id` values from a JSONL cache file."""

    completed: set[str] = set()
    if not jsonl_path.is_file():
        return completed
    with jsonl_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            uid = rec.get("unique_id")
            if uid is not None:
                completed.add(str(uid))
    return completed


def append_jsonl_records(jsonl_path: Path, records: Sequence[dict[str, Any]]) -> None:
    """Append records to a JSONL checkpoint file safely even after a truncated crash."""

    if not records:
        return
    jsonl_path.parent.mkdir(parents=True, exist_ok=True)
    needs_leading_newline = False
    if jsonl_path.is_file() and jsonl_path.stat().st_size > 0:
        with jsonl_path.open("rb") as rb:
            rb.seek(-1, 2)
            needs_leading_newline = rb.read(1) != b"\n"
    with jsonl_path.open("a", encoding="utf-8", newline="\n") as handle:
        if needs_leading_newline:
            handle.write("\n")
        for rec in records:
            handle.write(json.dumps(rec, ensure_ascii=False) + "\n")
        handle.flush()


def read_jsonl_records(jsonl_path: Path) -> list[dict[str, Any]]:
    """Load JSONL records deduplicated by `unique_id` (latest row wins)."""

    by_id: dict[str, dict[str, Any]] = {}
    if not jsonl_path.is_file():
        return []
    with jsonl_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            uid = rec.get("unique_id")
            if uid is not None:
                by_id[str(uid)] = rec
    return list(by_id.values())


def summarize_pilot_diagnostics(
    atom1_records: Sequence[dict[str, Any]],
    atom2_records: Sequence[dict[str, Any]],
    atom3_records: Sequence[dict[str, Any]],
    atom4_records: Sequence[dict[str, Any]],
    atom5_records: Sequence[dict[str, Any]],
) -> dict[str, Any]:
    """Compute Step 1B Pilot sanity metrics across Atom 1-5 caches."""

    def _by_uid(rows: Sequence[dict[str, Any]]) -> dict[str, dict[str, Any]]:
        return {str(r["unique_id"]): r for r in rows}

    a1 = _by_uid(atom1_records)
    a2 = _by_uid(atom2_records)
    a3 = _by_uid(atom3_records)
    a4 = _by_uid(atom4_records)
    a5 = _by_uid(atom5_records)

    def _atom_stats(rows: Sequence[dict[str, Any]], prefix: str = "") -> dict[str, Any]:
        if not rows:
            return {"count": 0}
        n = len(rows)
        statuses = {s: 0 for s in ("correct", "wrong", "invalid", "truncated")}
        for r in rows:
            st = str(r.get(f"{prefix}status", r.get("status", "wrong")))
            statuses[st] = statuses.get(st, 0) + 1
        out_toks = [float(r.get(f"{prefix}out_tokens", r.get("out_tokens", 0))) for r in rows]
        lats = [float(r.get(f"{prefix}latency_s", r.get("latency_s", 0.0))) for r in rows]
        return {
            "count": n,
            "accuracy": statuses["correct"] / n,
            "statuses": statuses,
            "truncated_rate": statuses["truncated"] / n,
            "invalid_rate": statuses["invalid"] / n,
            "mean_out_tokens": sum(out_toks) / n,
            "mean_latency_s": sum(lats) / n,
        }

    # Flip matrix on paired Atom 1 -> Atom 2
    common_12 = sorted(set(a1) & set(a2))
    flips = {"right_to_right": 0, "right_to_wrong": 0, "wrong_to_right": 0, "wrong_to_wrong": 0}
    for uid in common_12:
        c1 = bool(a1[uid]["correct"])
        c2 = bool(a2[uid]["refine_correct"])
        if c1 and c2:
            flips["right_to_right"] += 1
        elif c1 and not c2:
            flips["right_to_wrong"] += 1
        elif not c1 and c2:
            flips["wrong_to_right"] += 1
        else:
            flips["wrong_to_wrong"] += 1

    # Level breakdown for 1.7B vs 8B
    level_acc: dict[str, dict[str, float]] = {}
    for lv in range(1, 6):
        lv_1 = [r for r in atom1_records if int(r.get("level", 0)) == lv]
        lv_4 = [r for r in atom4_records if int(r.get("level", 0)) == lv]
        level_acc[f"level_{lv}"] = {
            "qwen3_1.7b_acc": (sum(bool(r["correct"]) for r in lv_1) / len(lv_1)) if lv_1 else 0.0,
            "qwen3_8b_acc": (sum(bool(r["correct"]) for r in lv_4) / len(lv_4)) if lv_4 else 0.0,
        }

    # Feature finite check on Atom 1
    feat_keys = (
        "mean_token_logprob",
        "min_token_logprob",
        "answer_logprob",
        "output_length",
        "format_valid",
        "question_length",
    )
    finite_ok = all(
        all(math.isfinite(float(r["features"][k])) for k in feat_keys) for r in atom1_records
    )

    # Atom 3 sample-0 accuracy sanity
    a3_s0_acc = (
        sum(bool(r["samples"][0]["correct"]) for r in atom3_records if r.get("samples"))
        / len(atom3_records)
        if atom3_records
        else 0.0
    )

    return {
        "atom1_1.7b_nonthink": _atom_stats(atom1_records),
        "atom2_1.7b_self_refine": _atom_stats(atom2_records, prefix="refine_"),
        "atom3_1.7b_sample0_acc": a3_s0_acc,
        "atom3_count": len(atom3_records),
        "atom4_8b_nonthink": _atom_stats(atom4_records),
        "atom5_1.7b_thinking": _atom_stats(atom5_records),
        "self_refine_flips": flips,
        "level_accuracy": level_acc,
        "features_all_finite": finite_ok,
        "common_12_count": len(common_12),
        "common_14_count": len(set(a1) & set(a4)),
        "common_15_count": len(set(a1) & set(a5)),
    }


def grade_and_check_sample(
    text: str,
    ground_truth: str,
    *,
    truncated: bool,
    strip_thinking: bool,
    think_closed: bool | None = None,
) -> dict[str, Any]:
    """Thin dict-returning wrapper around `classify_response` for JSONL serialization."""

    return classify_response(
        text,
        ground_truth,
        truncated=truncated,
        strip_thinking=strip_thinking,
        think_closed=think_closed,
    ).to_dict()


def split_stratified_halves(
    rows: Sequence[dict[str, Any]],
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    """Partition `rows` into two level-stratified halves (`core`, `extra`) deterministically."""

    by_level: dict[int, list[dict[str, Any]]] = {}
    for row in rows:
        lv = int(row.get("level", 0))
        by_level.setdefault(lv, []).append(row)

    core_uids: set[str] = set()
    for lv in sorted(by_level):
        grp = by_level[lv]
        half = len(grp) // 2
        core_uids.update(str(r["unique_id"]) for r in grp[:half])

    core = [r for r in rows if str(r["unique_id"]) in core_uids]
    extra = [r for r in rows if str(r["unique_id"]) not in core_uids]
    return core, extra


In [ ]:
# === Cell 4: Load or Deterministically Rebuild Zero-Leakage Splits + Verify SHA-256 ===
EXPECTED_SPLIT_HASHES = {
  "pilot_50.json": "779c13298481f9a36e1f924f600e901f4258cc94a623ea159466645f9ec74bb3",
  "router_train_600.json": "759c61fcf27678ff304230a86bf42f8c2f3e4d83dda028c49206ce5dcb061f30",
  "router_val_200.json": "6b1d7c1e0e43de43d60a64d1c69105e52c15e848be74970da6f6c6780418b067",
  "math500_test_500.json": "d398c9e8abbd4ce4c395ffda5cbd422a18235caea988634fc7fb2ce170fc143a",
  "splits_bundle.json": "ac7d6f81dc07dc7850374d3751f7e82cca9b8433cfa72e54e4b3d425beaa25d4"
}

SPLITS_DIR = OUTPUT_ROOT / "splits"
RAW_DIR = OUTPUT_ROOT / "raw"
BUNDLE_CANDIDATES = [
    SPLITS_DIR / "splits_bundle.json",
    Path("/content/splits_bundle.json"),
    Path("./data/splits/splits_bundle.json"),
]

bundle_loaded_from = None
for cand in BUNDLE_CANDIDATES:
    if cand.is_file() and sha256_file(cand) == EXPECTED_SPLIT_HASHES["splits_bundle.json"]:
        splits = json.loads(cand.read_text(encoding="utf-8"))
        bundle_loaded_from = cand
        print(f"Loaded verified splits_bundle.json from {cand}")
        break

if bundle_loaded_from is None:
    print("Downloading immutable official MATH sources & rebuilding deterministic splits...")
    raw_meta = ensure_raw_datasets(RAW_DIR, course_math500_path=None, force=False)
    raw_train = json.loads((RAW_DIR / "math_full_minus_math500.json").read_text(encoding="utf-8"))
    raw_test = json.loads((RAW_DIR / "math500_test.json").read_text(encoding="utf-8"))
    splits, manifest = build_splits(raw_train, raw_test, seed=SEED)
    write_splits(splits, manifest, SPLITS_DIR, raw_metadata=raw_meta)

    for fname, expected_sha in EXPECTED_SPLIT_HASHES.items():
        observed_sha = sha256_file(SPLITS_DIR / fname)
        assert observed_sha == expected_sha, (
            f"SHA-256 mismatch for {fname}: expected {expected_sha}, got {observed_sha}"
        )
    print("All split SHA-256 digests verified bit-for-bit against Step 1A manifest!")

for split_name, rows in splits.items():
    print(f"  - {split_name}: {len(rows)} problems")


In [ ]:
# === Cell 5: vLLM Continuous-Batch Engine, Exact Log-Probs & Single-GPU Calibration ===
import gc
import time
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

try:
    from vllm.distributed.parallel_state import destroy_distributed_environment, destroy_model_parallel
except ImportError:
    destroy_distributed_environment = None
    destroy_model_parallel = None


def load_vllm_model(
    model_id: str,
    *,
    candidates: list[tuple[bool, int, int, float]],
):
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False, token=False)
    stop_ids = set()
    if tokenizer.eos_token_id is not None:
        stop_ids.add(int(tokenizer.eos_token_id))
    for tok in ("<|im_end|>", "<|endoftext|>"):
        tid = tokenizer.convert_tokens_to_ids(tok)
        if isinstance(tid, int) and tid >= 0:
            stop_ids.add(tid)

    think_end_id = tokenizer.convert_tokens_to_ids("</think>")
    if not isinstance(think_end_id, int) or think_end_id < 0:
        think_end_id = 151668

    llm = None
    active_max_len = 0
    last_err = None
    for eager_flag, max_len, max_seqs, gpu_util in candidates:
        try:
            t0 = time.perf_counter()
            llm = LLM(
                model=model_id,
                dtype=DTYPE_STR,
                max_model_len=max_len,
                max_num_seqs=max_seqs,
                gpu_memory_utilization=gpu_util,
                enforce_eager=eager_flag,
                seed=SEED,
                trust_remote_code=False,
            )
            active_max_len = max_len
            print(
                f"Loaded {model_id} with vLLM (enforce_eager={eager_flag}, max_len={max_len}, "
                f"max_seqs={max_seqs}, util={gpu_util}) in {time.perf_counter() - t0:.1f}s"
            )
            break
        except Exception as exc:
            last_err = repr(exc)
            print(f"Load attempt (eager={eager_flag}, max_len={max_len}) failed: {last_err[:300]}")
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if llm is None:
        raise RuntimeError(f"Failed to load {model_id} in vLLM: {last_err}")

    return llm, tokenizer, stop_ids, think_end_id, active_max_len


def unload_vllm(llm_obj, tokenizer=None) -> None:
    try:
        engine = getattr(llm_obj, "llm_engine", None)
        core = getattr(engine, "engine_core", None)
        if core is not None and hasattr(core, "shutdown"):
            core.shutdown()
    except Exception:
        pass
    try:
        if destroy_model_parallel is not None:
            destroy_model_parallel()
        if destroy_distributed_environment is not None:
            destroy_distributed_environment()
    except Exception:
        pass
    del llm_obj
    if tokenizer is not None:
        del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    time.sleep(2.0)


def render_qwen3_prompt(tokenizer, messages: list[dict[str, str]], *, enable_thinking: bool) -> str:
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )


def _decode_completion_output(
    comp,
    tokenizer,
    stop_ids: set[int],
    think_end_id: int,
    *,
    allowed_max_tokens: int,
    collect_logprobs: bool,
    split_think_tokens: bool,
) -> dict[str, Any]:
    tids = list(comp.token_ids)
    ends_with_stop = bool(tids and tids[-1] in stop_ids)
    valid_ids = tids[:-1] if ends_with_stop else tids
    truncated = (comp.finish_reason == "length") or (not ends_with_stop and len(valid_ids) >= allowed_max_tokens)

    tok_logprobs: list[float] | None = None
    if collect_logprobs:
        raw_lps = comp.logprobs or []
        tok_logprobs = []
        for t_idx, tid in enumerate(valid_ids):
            if t_idx < len(raw_lps) and raw_lps[t_idx] is not None and tid in raw_lps[t_idx]:
                entry = raw_lps[t_idx][tid]
                val = float(getattr(entry, "logprob", entry))
                tok_logprobs.append(val if math.isfinite(val) else -20.0)
            else:
                tok_logprobs.append(-20.0)

    think_closed: bool | None = None
    think_tokens = 0
    body_text = ""
    full_text = tokenizer.decode(valid_ids, skip_special_tokens=True)

    if split_think_tokens:
        rev_pos = None
        for idx in range(len(valid_ids) - 1, -1, -1):
            if valid_ids[idx] == think_end_id:
                rev_pos = idx
                break
        if rev_pos is not None:
            think_closed = True
            think_tokens = rev_pos + 1
            body_text = tokenizer.decode(valid_ids[rev_pos + 1 :], skip_special_tokens=True).strip()
        else:
            think_closed = False
            think_tokens = len(valid_ids)
            body_text = ""

    return {
        "full_text": full_text,
        "body_text": body_text if split_think_tokens else full_text,
        "valid_ids": valid_ids,
        "out_tokens": len(valid_ids),
        "truncated": truncated,
        "token_logprobs": tok_logprobs,
        "think_closed": think_closed,
        "think_tokens": think_tokens,
    }


def generate_vllm_batch(
    llm,
    tokenizer,
    stop_ids: set[int],
    think_end_id: int,
    active_max_model_len: int,
    prompts: list[str],
    *,
    max_new_tokens: int,
    do_sample: bool = False,
    sampling_params: dict | None = None,
    n: int = 1,
    collect_logprobs: bool = False,
    split_think_tokens: bool = False,
    seed: int | None = None,
    use_tqdm: bool = True,
) -> list[Any]:
    prompt_id_lists = tokenizer(prompts, add_special_tokens=False)["input_ids"]
    allowed_list = [
        min(max_new_tokens, max(128, active_max_model_len - len(pids) - 16))
        for pids in prompt_id_lists
    ]

    base_kwargs: dict[str, Any] = {
        "n": n,
        "logprobs": 1 if collect_logprobs else None,
        "seed": seed if seed is not None else SEED,
    }
    if do_sample and sampling_params:
        base_kwargs.update(sampling_params)
    else:
        base_kwargs["temperature"] = 0.0

    if len(set(allowed_list)) == 1:
        sp_arg: Any = SamplingParams(max_tokens=allowed_list[0], **base_kwargs)
    else:
        sp_arg = [SamplingParams(max_tokens=amt, **base_kwargs) for amt in allowed_list]

    t0 = time.perf_counter()
    raw_outs = llm.generate(prompts, sp_arg, use_tqdm=use_tqdm)
    batch_elapsed = time.perf_counter() - t0
    per_item_latency = batch_elapsed / max(len(prompts), 1)
    per_path_latency = batch_elapsed / max(len(prompts) * n, 1)

    decoded: list[Any] = []
    for idx, out in enumerate(raw_outs):
        in_toks = len(out.prompt_token_ids) if getattr(out, "prompt_token_ids", None) else len(prompt_id_lists[idx])
        amt = allowed_list[idx]
        if n == 1:
            item = _decode_completion_output(
                out.outputs[0],
                tokenizer,
                stop_ids,
                think_end_id,
                allowed_max_tokens=amt,
                collect_logprobs=collect_logprobs,
                split_think_tokens=split_think_tokens,
            )
            item["in_tokens"] = in_toks
            item["batch_latency_s"] = per_item_latency
            decoded.append(item)
        else:
            paths = []
            for comp in out.outputs:
                item = _decode_completion_output(
                    comp,
                    tokenizer,
                    stop_ids,
                    think_end_id,
                    allowed_max_tokens=amt,
                    collect_logprobs=collect_logprobs,
                    split_think_tokens=split_think_tokens,
                )
                item["in_tokens"] = in_toks
                item["batch_latency_s"] = per_path_latency
                paths.append(item)
            decoded.append(paths)
    return decoded


def calibrate_bs1_latency(
    llm,
    tokenizer,
    stop_ids: set[int],
    think_end_id: int,
    active_max_model_len: int,
    rows: list[dict[str, Any]],
    *,
    mode_label: str,
    enable_thinking: bool,
    max_new_tokens: int,
    do_sample: bool,
    sampling_params: dict | None,
    out_jsonl: Path,
    limit: int = 10,
) -> dict[str, Any]:
    """Measure true single-request (bs=1) end-to-end latency on the active GPU and engine."""

    done_keys = set()
    if out_jsonl.is_file():
        for line in out_jsonl.read_text(encoding="utf-8").splitlines():
            if line.strip():
                try:
                    rec = json.loads(line)
                    if rec.get("gpu_name") == gpu_name and rec.get("engine") == ENGINE_NAME:
                        done_keys.add((rec["mode_label"], str(rec["unique_id"])))
                except Exception:
                    pass

    sample_rows = rows[:limit]
    pending_cal = [
        (idx, row)
        for idx, row in enumerate(sample_rows)
        if (mode_label, str(row["unique_id"])) not in done_keys
    ]
    print(
        f"[Latency Calibration | {mode_label} | {gpu_name} | {ENGINE_NAME}] "
        f"pending={len(pending_cal)}/{len(sample_rows)} (bs=1)...",
        flush=True,
    )

    if pending_cal and not any(k[0] == mode_label for k in done_keys):
        warm_p = render_qwen3_prompt(
            tokenizer, build_solve_messages(sample_rows[0]["problem"]), enable_thinking=enable_thinking
        )
        _ = generate_vllm_batch(
            llm, tokenizer, stop_ids, think_end_id, active_max_model_len, [warm_p],
            max_new_tokens=min(64, max_new_tokens),
            do_sample=do_sample, sampling_params=sampling_params,
            collect_logprobs=False, split_think_tokens=enable_thinking,
            seed=SEED, use_tqdm=False,
        )

    for idx, row in tqdm(pending_cal, desc=f"Calibrate bs=1 {mode_label}"):
        prompt = render_qwen3_prompt(
            tokenizer, build_solve_messages(row["problem"]), enable_thinking=enable_thinking
        )
        res = generate_vllm_batch(
            llm, tokenizer, stop_ids, think_end_id, active_max_model_len, [prompt],
            max_new_tokens=max_new_tokens,
            do_sample=do_sample, sampling_params=sampling_params,
            collect_logprobs=False, split_think_tokens=enable_thinking,
            seed=SEED + idx, use_tqdm=False,
        )[0]
        rec = {
            "mode_label": mode_label,
            "unique_id": row["unique_id"],
            "level": row["level"],
            "in_tokens": res["in_tokens"],
            "out_tokens": res["out_tokens"],
            "bs1_latency_s": res["batch_latency_s"],
            "truncated": res["truncated"],
            "gpu_name": gpu_name,
            "engine": ENGINE_NAME,
            "max_new_tokens": max_new_tokens,
        }
        append_jsonl_records(out_jsonl, [rec])

    all_mode = []
    if out_jsonl.is_file():
        for line in out_jsonl.read_text(encoding="utf-8").splitlines():
            if line.strip():
                rec = json.loads(line)
                if (
                    rec.get("mode_label") == mode_label
                    and rec.get("gpu_name") == gpu_name
                    and rec.get("engine") == ENGINE_NAME
                ):
                    all_mode.append(rec)
    total_tok = sum(r["out_tokens"] for r in all_mode)
    total_sec = sum(r["bs1_latency_s"] for r in all_mode)
    tok_per_s = total_tok / total_sec if total_sec > 0 else 0.0
    print(f"[Latency Calibration | {mode_label} | {gpu_name} | {ENGINE_NAME}] n={len(all_mode)} | {tok_per_s:.1f} tok/s (bs=1)")
    return {"mode_label": mode_label, "count": len(all_mode), "bs1_tok_per_s": tok_per_s}


In [ ]:
# === Cell 6: Resumable Priority Queue with Stratified Incremental Checkpoints ===
from tqdm.auto import tqdm


def _chunked(seq: list, size: int):
    for i in range(0, len(seq), size):
        yield seq[i : i + size]


def run_atom1_first_pass(
    llm, tokenizer, stop_ids, think_end_id, max_len, rows, out_path: Path, *, chunk_size: int, label: str = ""
) -> list[dict[str, Any]]:
    done = load_completed_ids(out_path)
    pending = [r for r in rows if str(r["unique_id"]) not in done]
    tag = label or out_path.parent.name
    print(f"[Atom 1 | 1.7B Non-Think] {tag}: pending={len(pending)}/{len(rows)}")
    decode_fn = lambda ids: tokenizer.decode(ids, skip_special_tokens=True)

    for chunk in _chunked(pending, chunk_size):
        prompts = [
            render_qwen3_prompt(tokenizer, build_solve_messages(r["problem"]), enable_thinking=False)
            for r in chunk
        ]
        outs = generate_vllm_batch(
            llm, tokenizer, stop_ids, think_end_id, max_len, prompts,
            max_new_tokens=MAX_NEW_TOKENS_NON_THINK,
            do_sample=False, collect_logprobs=True, split_think_tokens=False,
        )
        records = []
        for r, out in zip(chunk, outs):
            grade = grade_and_check_sample(
                out["body_text"], r["answer"], truncated=out["truncated"], strip_thinking=False
            )
            feats = extract_router_features(
                out["token_logprobs"], out["valid_ids"], decode_fn,
                input_tokens=out["in_tokens"], truncated=out["truncated"],
                response_text=out["body_text"],
            )
            records.append(
                {
                    "unique_id": r["unique_id"],
                    "split_name": r["split_name"],
                    "level": r["level"],
                    "subject": r["subject"],
                    "problem": r["problem"],
                    "ground_truth": r["answer"],
                    "response_text": out["body_text"],
                    "in_tokens": out["in_tokens"],
                    "out_tokens": out["out_tokens"],
                    "latency_s": out["batch_latency_s"],
                    "features": feats,
                    **grade,
                }
            )
        append_jsonl_records(out_path, records)
    return read_jsonl_records(out_path)


def run_atom2_self_refine(
    llm, tokenizer, stop_ids, think_end_id, max_len, atom1_rows: list[dict[str, Any]], out_path: Path, *, chunk_size: int
) -> list[dict[str, Any]]:
    done = load_completed_ids(out_path)
    pending = [r for r in atom1_rows if str(r["unique_id"]) not in done]
    print(f"[Atom 2 | 1.7B Self-Refine] {out_path.parent.name}: pending={len(pending)}/{len(atom1_rows)}")

    for chunk in _chunked(pending, chunk_size):
        crit_prompts = [
            render_qwen3_prompt(
                tokenizer,
                build_critique_messages(r["problem"], r["response_text"]),
                enable_thinking=False,
            )
            for r in chunk
        ]
        crit_outs = generate_vllm_batch(
            llm, tokenizer, stop_ids, think_end_id, max_len, crit_prompts,
            max_new_tokens=MAX_NEW_TOKENS_CRITIQUE,
            do_sample=False, collect_logprobs=False, split_think_tokens=False,
        )
        ref_prompts = [
            render_qwen3_prompt(
                tokenizer,
                build_refine_messages(r["problem"], r["response_text"], c["body_text"]),
                enable_thinking=False,
            )
            for r, c in zip(chunk, crit_outs)
        ]
        ref_outs = generate_vllm_batch(
            llm, tokenizer, stop_ids, think_end_id, max_len, ref_prompts,
            max_new_tokens=MAX_NEW_TOKENS_NON_THINK,
            do_sample=False, collect_logprobs=False, split_think_tokens=False,
        )
        records = []
        for r, c, rf in zip(chunk, crit_outs, ref_outs):
            grade = grade_and_check_sample(
                rf["body_text"], r["ground_truth"], truncated=rf["truncated"], strip_thinking=False
            )
            records.append(
                {
                    "unique_id": r["unique_id"],
                    "split_name": r["split_name"],
                    "level": r["level"],
                    "critique_text": c["body_text"],
                    "critique_in_tokens": c["in_tokens"],
                    "critique_out_tokens": c["out_tokens"],
                    "critique_truncated": c["truncated"],
                    "critique_latency_s": c["batch_latency_s"],
                    "refine_text": rf["body_text"],
                    "refine_in_tokens": rf["in_tokens"],
                    "refine_out_tokens": rf["out_tokens"],
                    "refine_latency_s": rf["batch_latency_s"],
                    "refine_status": grade["status"],
                    "refine_correct": grade["correct"],
                    "refine_extracted_answer": grade["extracted_answer"],
                    "refine_has_boxed": grade["has_boxed"],
                    "refine_truncated": grade["truncated"],
                }
            )
        append_jsonl_records(out_path, records)
    return read_jsonl_records(out_path)


def run_atom3_sampling(
    llm, tokenizer, stop_ids, think_end_id, max_len, rows, out_path: Path, *, chunk_size: int
) -> list[dict[str, Any]]:
    done = load_completed_ids(out_path)
    pending = [r for r in rows if str(r["unique_id"]) not in done]
    print(f"[Atom 3 | 1.7B {N_SAMPLING_PATHS}-Path Shared-Prefix] {out_path.parent.name}: pending={len(pending)}/{len(rows)}")

    for chunk_idx, chunk in enumerate(_chunked(pending, chunk_size)):
        prompts = [
            render_qwen3_prompt(tokenizer, build_solve_messages(r["problem"]), enable_thinking=False)
            for r in chunk
        ]
        outs_per_prompt = generate_vllm_batch(
            llm, tokenizer, stop_ids, think_end_id, max_len, prompts,
            max_new_tokens=MAX_NEW_TOKENS_NON_THINK,
            do_sample=True, sampling_params=SAMPLING_PARAMS_NON_THINK,
            n=N_SAMPLING_PATHS, collect_logprobs=False, split_think_tokens=False,
            seed=SEED + chunk_idx,
        )
        records = []
        for r, path_outs in zip(chunk, outs_per_prompt):
            samples = []
            for path_idx, out in enumerate(path_outs):
                grade = grade_and_check_sample(
                    out["body_text"], r["answer"], truncated=out["truncated"], strip_thinking=False
                )
                samples.append(
                    {
                        "sample_idx": path_idx,
                        "response_text": out["body_text"],
                        "in_tokens": out["in_tokens"],
                        "out_tokens": out["out_tokens"],
                        "latency_s": out["batch_latency_s"],
                        **grade,
                    }
                )
            records.append(
                {
                    "unique_id": r["unique_id"],
                    "split_name": r["split_name"],
                    "level": r["level"],
                    "ground_truth": r["answer"],
                    "samples": samples,
                }
            )
        append_jsonl_records(out_path, records)
    return read_jsonl_records(out_path)


def run_atom4_large_model(
    llm, tokenizer, stop_ids, think_end_id, max_len, rows, out_path: Path, *, chunk_size: int, label: str = ""
) -> list[dict[str, Any]]:
    done = load_completed_ids(out_path)
    pending = [r for r in rows if str(r["unique_id"]) not in done]
    tag = label or out_path.parent.name
    print(f"[Atom 4 | 8B Non-Think] {tag}: pending={len(pending)}/{len(rows)}")

    for chunk in _chunked(pending, chunk_size):
        prompts = [
            render_qwen3_prompt(tokenizer, build_solve_messages(r["problem"]), enable_thinking=False)
            for r in chunk
        ]
        outs = generate_vllm_batch(
            llm, tokenizer, stop_ids, think_end_id, max_len, prompts,
            max_new_tokens=MAX_NEW_TOKENS_NON_THINK,
            do_sample=False, collect_logprobs=False, split_think_tokens=False,
        )
        records = []
        for r, out in zip(chunk, outs):
            grade = grade_and_check_sample(
                out["body_text"], r["answer"], truncated=out["truncated"], strip_thinking=False
            )
            records.append(
                {
                    "unique_id": r["unique_id"],
                    "split_name": r["split_name"],
                    "level": r["level"],
                    "ground_truth": r["answer"],
                    "response_text": out["body_text"],
                    "in_tokens": out["in_tokens"],
                    "out_tokens": out["out_tokens"],
                    "latency_s": out["batch_latency_s"],
                    **grade,
                }
            )
        append_jsonl_records(out_path, records)
    return read_jsonl_records(out_path)


def run_atom5_thinking(
    llm, tokenizer, stop_ids, think_end_id, max_len, rows, out_path: Path, *, chunk_size: int
) -> list[dict[str, Any]]:
    done = load_completed_ids(out_path)
    pending = [r for r in rows if str(r["unique_id"]) not in done]
    print(f"[Atom 5 | 1.7B Thinking] {out_path.parent.name}: pending={len(pending)}/{len(rows)}")

    for chunk_idx, chunk in enumerate(_chunked(pending, chunk_size)):
        prompts = [
            render_qwen3_prompt(tokenizer, build_solve_messages(r["problem"]), enable_thinking=True)
            for r in chunk
        ]
        outs = generate_vllm_batch(
            llm, tokenizer, stop_ids, think_end_id, max_len, prompts,
            max_new_tokens=MAX_NEW_TOKENS_THINK,
            do_sample=True, sampling_params=SAMPLING_PARAMS_THINK,
            collect_logprobs=False, split_think_tokens=True, seed=SEED + chunk_idx,
        )
        records = []
        for r, out in zip(chunk, outs):
            grade = grade_and_check_sample(
                out["body_text"],
                r["answer"],
                truncated=out["truncated"],
                strip_thinking=True,
                think_closed=out["think_closed"],
            )
            records.append(
                {
                    "unique_id": r["unique_id"],
                    "split_name": r["split_name"],
                    "level": r["level"],
                    "ground_truth": r["answer"],
                    "full_text": out["full_text"],
                    "final_body": out["body_text"],
                    "in_tokens": out["in_tokens"],
                    "out_tokens": out["out_tokens"],
                    "think_tokens": out["think_tokens"],
                    "latency_s": out["batch_latency_s"],
                    **grade,
                }
            )
        append_jsonl_records(out_path, records)
    return read_jsonl_records(out_path)


# --- Build Stratified Execution Queue ---
if RUN_MODE == "pilot":
    EXEC_QUEUE = [
        {"label": "pilot_50_v2", "rows": splits["pilot_50"], "out_dir": f"pilot_50_{RUN_TAG}", "atoms": (1, 2, 3, 4, 5)},
    ]
    SUMMARY_SPECS = [
        {"out_dir": f"pilot_50_{RUN_TAG}", "atoms": (1, 2, 3, 4, 5)},
    ]
elif RUN_MODE == "full":
    train_core_300, train_extra_300 = split_stratified_halves(splits["router_train_600"])
    EXEC_QUEUE = [
        {"label": "math500_test_500", "rows": splits["math500_test_500"], "out_dir": "math500_test_500", "atoms": (1, 2, 3, 4, 5)},
        {"label": "router_val_200", "rows": splits["router_val_200"], "out_dir": "router_val_200", "atoms": (1, 4)},
        {"label": "router_train_core_300", "rows": train_core_300, "out_dir": "router_train_600", "atoms": (1, 4)},
        {"label": "router_train_extra_300", "rows": train_extra_300, "out_dir": "router_train_600", "atoms": (1, 4)},
    ]
    SUMMARY_SPECS = [
        {"out_dir": "math500_test_500", "atoms": (1, 2, 3, 4, 5)},
        {"out_dir": "router_val_200", "atoms": (1, 4)},
        {"out_dir": "router_train_600", "atoms": (1, 4)},
    ]
else:
    raise ValueError(f"Unknown RUN_MODE={RUN_MODE!r}")

run_started = time.perf_counter()
cal_path = OUTPUT_ROOT / f"latency_calibration_{RUN_TAG}.jsonl"

# --- Stage A: Qwen3-1.7B Non-Thinking (Atoms 1, 2, 3) ---
print(f"\nLoading {MODEL_SMALL} (Stage A: Non-Thinking Atoms 1-3)...")
llm_small, tok_small, stop_small, think_end_small, max_len_small = load_vllm_model(
    MODEL_SMALL,
    candidates=[(False, 8192, 64, 0.85), (False, 6144, 32, 0.85), (True, 6144, 32, 0.85)],
)

if RUN_BS1_LATENCY_CALIBRATION:
    calibrate_bs1_latency(
        llm_small, tok_small, stop_small, think_end_small, max_len_small, splits["pilot_50"],
        mode_label="qwen3_1.7b_non_think", enable_thinking=False,
        max_new_tokens=MAX_NEW_TOKENS_NON_THINK, do_sample=False, sampling_params=None,
        out_jsonl=cal_path, limit=CALIBRATION_SAMPLE_COUNT,
    )

for job in EXEC_QUEUE:
    sdir = OUTPUT_ROOT / job["out_dir"]
    rows = job["rows"]
    atoms = set(job["atoms"])
    a1 = []
    if 1 in atoms:
        a1 = run_atom1_first_pass(
            llm_small, tok_small, stop_small, think_end_small, max_len_small,
            rows, sdir / "atom1_1.7b_first.jsonl", chunk_size=CHUNK_SIZE_SMALL, label=job["label"],
        )
    if 2 in atoms:
        a1_by_uid = {str(r["unique_id"]): r for r in a1}
        ordered_a1 = [a1_by_uid[str(r["unique_id"])] for r in rows if str(r["unique_id"]) in a1_by_uid]
        run_atom2_self_refine(
            llm_small, tok_small, stop_small, think_end_small, max_len_small,
            ordered_a1, sdir / "atom2_1.7b_self_refine.jsonl", chunk_size=CHUNK_SIZE_SMALL,
        )
    if 3 in atoms:
        run_atom3_sampling(
            llm_small, tok_small, stop_small, think_end_small, max_len_small,
            rows, sdir / "atom3_1.7b_samples3.jsonl", chunk_size=CHUNK_SIZE_SMALL,
        )

unload_vllm(llm_small, tok_small)

# --- Stage B: Qwen3-8B Non-Thinking (Atom 4, runs BEFORE slow Thinking) ---
print(f"\nLoading {MODEL_LARGE} (Stage B: 8B Atom 4)...")
llm_large, tok_large, stop_large, think_end_large, max_len_large = load_vllm_model(
    MODEL_LARGE,
    candidates=[(False, 4096, 32, 0.92), (False, 3072, 32, 0.90), (True, 3072, 32, 0.92), (True, 2560, 16, 0.92)],
)

if RUN_BS1_LATENCY_CALIBRATION:
    calibrate_bs1_latency(
        llm_large, tok_large, stop_large, think_end_large, max_len_large, splits["pilot_50"],
        mode_label="qwen3_8b_non_think", enable_thinking=False,
        max_new_tokens=MAX_NEW_TOKENS_NON_THINK, do_sample=False, sampling_params=None,
        out_jsonl=cal_path, limit=CALIBRATION_SAMPLE_COUNT,
    )

for job in EXEC_QUEUE:
    if 4 in job["atoms"]:
        sdir = OUTPUT_ROOT / job["out_dir"]
        run_atom4_large_model(
            llm_large, tok_large, stop_large, think_end_large, max_len_large,
            job["rows"], sdir / "atom4_8b_first.jsonl", chunk_size=CHUNK_SIZE_LARGE, label=job["label"],
        )

unload_vllm(llm_large, tok_large)

# --- Stage C: Reload cached Qwen3-1.7B for Slow Thinking (Atom 5) LAST ---
if any(5 in job["atoms"] for job in EXEC_QUEUE):
    print(f"\nLoading {MODEL_SMALL} (Stage C: Slow Thinking Atom 5)...")
    llm_small, tok_small, stop_small, think_end_small, max_len_small = load_vllm_model(
        MODEL_SMALL,
        candidates=[(False, 6144, 64, 0.85), (True, 6144, 32, 0.85)],
    )
    if RUN_BS1_LATENCY_CALIBRATION:
        calibrate_bs1_latency(
            llm_small, tok_small, stop_small, think_end_small, max_len_small, splits["pilot_50"],
            mode_label="qwen3_1.7b_think", enable_thinking=True,
            max_new_tokens=MAX_NEW_TOKENS_THINK, do_sample=True, sampling_params=SAMPLING_PARAMS_THINK,
            out_jsonl=cal_path, limit=CALIBRATION_THINK_COUNT,
        )
    for job in EXEC_QUEUE:
        if 5 in job["atoms"]:
            sdir = OUTPUT_ROOT / job["out_dir"]
            run_atom5_thinking(
                llm_small, tok_small, stop_small, think_end_small, max_len_small,
                job["rows"], sdir / "atom5_1.7b_thinking.jsonl", chunk_size=CHUNK_SIZE_SMALL,
            )
    unload_vllm(llm_small, tok_small)

total_elapsed_min = (time.perf_counter() - run_started) / 60.0
print(f"\nFinished RUN_MODE={RUN_MODE!r} ({RUN_TAG}, engine={ENGINE_NAME}) in {total_elapsed_min:.2f} min.")


In [ ]:
# === Cell 7: Diagnostics Summary & Zip Bundle Export ===
import zipfile

all_summaries = {}
for spec in SUMMARY_SPECS:
    sdir = OUTPUT_ROOT / spec["out_dir"]
    atoms = set(spec["atoms"])
    a1 = read_jsonl_records(sdir / "atom1_1.7b_first.jsonl") if 1 in atoms else []
    a2 = read_jsonl_records(sdir / "atom2_1.7b_self_refine.jsonl") if 2 in atoms else []
    a3 = read_jsonl_records(sdir / "atom3_1.7b_samples3.jsonl") if 3 in atoms else []
    a4 = read_jsonl_records(sdir / "atom4_8b_first.jsonl") if 4 in atoms else []
    a5 = read_jsonl_records(sdir / "atom5_1.7b_thinking.jsonl") if 5 in atoms else []
    diag = summarize_pilot_diagnostics(a1, a2, a3, a4, a5)
    if a3:
        distinct_counts = [len({s["response_text"] for s in r.get("samples", [])}) for r in a3]
        diag["atom3_all_3_distinct"] = sum(c == N_SAMPLING_PATHS for c in distinct_counts)
    diag["elapsed_minutes"] = round(total_elapsed_min, 2)
    diag["gpu_name"] = gpu_name
    diag["engine"] = ENGINE_NAME
    diag["max_new_tokens_non_think"] = MAX_NEW_TOKENS_NON_THINK
    diag["max_new_tokens_critique"] = MAX_NEW_TOKENS_CRITIQUE
    all_summaries[spec["out_dir"]] = diag

    summary_path = sdir / "summary.json"
    summary_path.write_text(json.dumps(diag, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"\n===== SUMMARY: {spec['out_dir']} =====")
    print(json.dumps(diag, ensure_ascii=False, indent=2))

zip_name = f"math550_{RUN_MODE}_{RUN_TAG}_bundle.zip"
zip_path = OUTPUT_ROOT / zip_name
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for spec in SUMMARY_SPECS:
        sdir = OUTPUT_ROOT / spec["out_dir"]
        for p in sorted(sdir.glob("*")):
            if p.is_file():
                zf.write(p, arcname=f"{spec['out_dir']}/{p.name}")
    if cal_path.is_file():
        zf.write(cal_path, arcname=cal_path.name)

print(f"\nPackaged bundle ready: {zip_path} ({zip_path.stat().st_size / 1024:.1f} KB)")
if HAS_COLAB_PKG:
    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception as exc:
        print(f"Auto-download skipped ({exc}); download {zip_path} manually from sidebar or Drive.")
